# Complete Blood Donor Retention Pipeline (Colab Edition)

This notebook runs the entire machine learning pipeline end-to-end.

**Changes made for Colab & Correctness:**
- Consolidated all modules (`src/`) into this single notebook.
- **Fixed Longitudinal Data Leakage:** Changed random splitting to Group-based splitting using `Donor_ID` to prevent the model from evaluating on donors it saw in the training set.
- **Fixed Feature Leakage:** Removed `Donation_Frequency_Label` from the point-in-time features, as it represents future information from the master table.
- Added native categorical handling for tree models.


In [ ]:
!pip install -q pandas numpy scikit-learn xgboost lightgbm catboost shap matplotlib seaborn

import os
from pathlib import Path
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from google.colab import files
import io

# Setup Directories
OUTPUT_DIR = Path('/content/outputs')
FIGURES_DIR = OUTPUT_DIR / 'figures'
MODELS_DIR = OUTPUT_DIR / 'models'
METRICS_DIR = OUTPUT_DIR / 'metrics'
for d in [FIGURES_DIR, MODELS_DIR, METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Please upload 'Samarpan_BloodBank_SyntheticDataset_V2.xlsx'")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
print(f"Loaded {filename}")


In [ ]:
def _standardize_text(series):
    return series.astype(str).str.strip().str.replace(r"\s+", " ", regex=True)

def clean_data(excel_bytes):
    donors = pd.read_excel(excel_bytes, sheet_name="Donor_Master")
    donations = pd.read_excel(excel_bytes, sheet_name="Donation_Register")
    
    donors['Registration_Date'] = pd.to_datetime(donors['Registration_Date'], errors='coerce')
    donors = donors.drop_duplicates(subset=["Donor_ID"]).copy()
    for col in ["Gender", "Blood_Group", "Donation_Frequency", "Donor_Status"]:
        if col in donors.columns: donors[col] = _standardize_text(donors[col])
    donors["Age"] = pd.to_numeric(donors["Age"], errors="coerce")
    
    donations['Donation_Date'] = pd.to_datetime(donations['Donation_Date'], errors='coerce')
    donations = donations.drop_duplicates(subset=["Donation_ID"]).copy()
    for col in ["Blood_Group", "Donation_Type", "Collection_Source"]:
        if col in donations.columns: donations[col] = _standardize_text(donations[col])
    donations["Units_Collected"] = pd.to_numeric(donations["Units_Collected"], errors="coerce")
    donations = donations.sort_values(["Donor_ID", "Donation_Date", "Donation_ID"]).reset_index(drop=True)
    
    # Align
    valid_ids = set(donors["Donor_ID"])
    donations = donations[donations["Donor_ID"].isin(valid_ids)].copy()
    
    return donors.reset_index(drop=True), donations.reset_index(drop=True)

donors_df, donations_df = clean_data(io.BytesIO(uploaded[filename]))
print(f"Donors: {len(donors_df)}, Donations: {len(donations_df)}")


In [ ]:
RETENTION_DAYS = 180
CHURN_DAYS = 365
AGE_BINS = [0, 25, 35, 45, 60, 120]
AGE_LABELS = ["18-25", "26-35", "36-45", "46-60", "60+"]

def _count_in_window(dates, anchor, days):
    start = anchor - pd.Timedelta(days=days)
    return int(((dates > start) & (dates <= anchor)).sum())

def build_observation_dataset(donors, donations):
    donors_idx = donors.set_index("Donor_ID")
    max_date = donations["Donation_Date"].max()
    records = []

    for donor_id, donor_donations in donations.groupby("Donor_ID"):
        donor_donations = donor_donations.sort_values("Donation_Date").reset_index(drop=True)
        donor = donors_idx.loc[donor_id]
        reg_date = donor["Registration_Date"]
        future_dates = donor_donations["Donation_Date"].tolist()

        for idx, row in donor_donations.iterrows():
            anchor = row["Donation_Date"]
            history = donor_donations.iloc[: idx + 1]
            h_dates = history["Donation_Date"]
            
            retained_end = anchor + pd.Timedelta(days=RETENTION_DAYS)
            churn_end = anchor + pd.Timedelta(days=CHURN_DAYS)
            future_after = [d for d in future_dates if d > anchor]

            retained_180, churn = np.nan, np.nan
            if retained_end <= max_date:
                retained_180 = int(any(anchor < d <= retained_end for d in future_after))
            if churn_end <= max_date:
                churn = int(not any(anchor < d <= churn_end for d in future_after))

            total_dons = len(history)
            tenure_days = (anchor - reg_date).days
            
            # Gaps
            if len(h_dates) > 1:
                gaps = h_dates.sort_values().diff().dt.days.dropna()
                avg_gap = gaps.mean()
                days_since_last = int((anchor - h_dates.iloc[-2]).days)
            else:
                avg_gap = -1.0 # Fix: use out-of-bounds for first-timers
                days_since_last = -1
                
            camp_count = int((history["Collection_Source"] == "Camp").sum())
            walkin_count = int((history["Collection_Source"] == "Walk-in").sum())

            records.append({
                "Donor_ID": donor_id,
                "anchor_date": anchor,
                "Gender": donor["Gender"],
                "Age": donor["Age"],
                "Blood_Group": donor["Blood_Group"],
                "days_since_registration": max(tenure_days, 0),
                "days_since_last_donation": days_since_last,
                "total_donations": total_dons,
                "donations_last_6_months": _count_in_window(h_dates, anchor, 180),
                "avg_gap_days": avg_gap,
                "camp_ratio": camp_count / total_dons,
                "total_units_donated": float(history["Units_Collected"].sum()),
                "is_first_donation": int(total_dons == 1),
                "retained_180": retained_180,
                "churn_365": churn,
            })

    df = pd.DataFrame(records)
    df["age_group"] = pd.cut(df["Age"], bins=AGE_BINS, labels=AGE_LABELS, right=True, include_lowest=True).astype(str)
    return df

dataset = build_observation_dataset(donors_df, donations_df)
print(f"Observation Rows: {len(dataset)}")


In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, precision_score, recall_score, f1_score

def prepare_data(dataset, target_col):
    labeled = dataset.dropna(subset=[target_col]).copy()
    features = ["days_since_registration", "days_since_last_donation", "total_donations", 
                "donations_last_6_months", "avg_gap_days", "camp_ratio", "total_units_donated", 
                "is_first_donation", "Age", "Gender", "Blood_Group", "age_group"]
    
    x = labeled[features].copy()
    y = labeled[target_col].astype(int)
    groups = labeled["Donor_ID"]
    return x, y, groups

def build_preprocessor(x):
    cat_features = ["Gender", "Blood_Group", "age_group"]
    num_features = [c for c in x.columns if c not in cat_features]
    
    return ColumnTransformer([
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_features),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore"))]), cat_features),
    ])

def train_and_evaluate(x, y, groups, target_name):
    # FIXED: GroupShuffleSplit prevents leakage of the same donor into train and test
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(x, y, groups))
    x_train, x_test = x.iloc[train_idx], x.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    models = {
        "RandomForest": RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42, n_jobs=-1),
        "XGBoost": XGBClassifier(n_estimators=100, learning_rate=0.05, eval_metric="logloss", random_state=42, n_jobs=-1),
        "LightGBM": LGBMClassifier(n_estimators=100, learning_rate=0.05, class_weight="balanced", random_state=42, n_jobs=-1, verbose=-1)
    }
    
    preprocessor = build_preprocessor(x)
    results = []
    best_pipeline = None
    best_auc = 0
    
    for name, estimator in models.items():
        pipeline = Pipeline([("preprocessor", preprocessor), ("model", estimator)])
        pipeline.fit(x_train, y_train)
        
        y_prob = pipeline.predict_proba(x_test)[:, 1]
        y_pred = pipeline.predict(x_test)
        
        auc = roc_auc_score(y_test, y_prob)
        if auc > best_auc:
            best_auc = auc
            best_pipeline = pipeline
            
        results.append({
            "Target": target_name,
            "Model": name,
            "ROC-AUC": auc,
            "PR-AUC": average_precision_score(y_test, y_prob),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1": f1_score(y_test, y_pred, zero_division=0)
        })
        
    return pd.DataFrame(results), best_pipeline, x_test


In [ ]:
# Train Models
res_retention, ret_model, x_test_ret = train_and_evaluate(*prepare_data(dataset, "retained_180"), "180-Day Retention")
res_churn, churn_model, x_test_churn = train_and_evaluate(*prepare_data(dataset, "churn_365"), "365-Day Churn")

display(pd.concat([res_retention, res_churn]).sort_values(["Target", "ROC-AUC"], ascending=[True, False]).round(3))

# SHAP Explainability for Churn Model
print("\nGenerating SHAP Summary for Churn Model...")
preprocessor = churn_model.named_steps["preprocessor"]
model = churn_model.named_steps["model"]
x_transformed = preprocessor.transform(x_test_churn)
feature_names = preprocessor.get_feature_names_out()

# Sample for SHAP
sample_idx = np.random.choice(x_transformed.shape[0], min(500, x_transformed.shape[0]), replace=False)
x_sample = x_transformed[sample_idx]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(x_sample)
if isinstance(shap_values, list): shap_values = shap_values[1] # For some RF variants

shap.summary_plot(shap_values, features=x_sample, feature_names=feature_names, max_display=15)


In [ ]:
def build_rfm_table(donors, donations):
    ref = donations["Donation_Date"].max()
    recency = donations.groupby("Donor_ID")["Donation_Date"].max().rsub(ref).dt.days.rename("recency")
    freq = donations.groupby("Donor_ID").size().rename("freq")
    vol = donations.groupby("Donor_ID")["Units_Collected"].sum().rename("units")
    
    rfm = pd.concat([recency, freq, vol], axis=1).reset_index()
    for col in ["recency", "freq", "units"]:
        rfm[f"{col}_score"] = pd.qcut(rfm[col], 4, labels=[1,2,3,4], duplicates="drop").astype(int)
        
    def assign_segment(row):
        if row["freq_score"] >= 3 and row["recency_score"] >= 3: return "Loyal"
        if row["recency_score"] >= 3: return "Active"
        if row["recency_score"] == 2: return "At-Risk"
        return "Lost"
        
    rfm["Segment"] = rfm.apply(assign_segment, axis=1)
    return rfm

rfm_df = build_rfm_table(donors_df, donations_df)

# Score active donors using true model
latest = dataset.sort_values("anchor_date").groupby("Donor_ID", as_index=False).tail(1)
x_latest, _, _ = prepare_data(latest.assign(retained_180=1), "retained_180") # dummy target to reuse logic
probs = ret_model.predict_proba(x_latest)[:, 1]

plan = latest[["Donor_ID"]].copy()
plan["Retention_Probability"] = probs
plan = plan.merge(rfm_df[["Donor_ID", "Segment"]], on="Donor_ID", how="left")

def recommend(row):
    p = row["Retention_Probability"]
    if p < 0.50: return "SMS reminder / Phone outreach"
    elif p < 0.80: return "Personalized invitation"
    return "Loyalty appreciation"

plan["Intervention"] = plan.apply(recommend, axis=1)

print("🔥 TOP 10 DONORS REQUIRING INTERVENTION TODAY 🔥")
plan = plan.sort_values("Retention_Probability").head(10)
plan["Retention_Probability"] = plan["Retention_Probability"].apply(lambda x: f"{x:.1%}")
display(plan)
